# 572. Subtree of Another Tree
**Difficulty:** 🟢 Easy · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/subtree-of-another-tree/

## 💡 Concepts

**Core concept(s):** Reuse a **"same tree?"** check at every node; or turn both trees into **strings** and check substring.

**Why it applies here:** `sub` is a subtree of `root` if, at some node of `root`, the tree hanging there is identical to `sub`. The clever version flattens both trees to text so "is it somewhere inside?" becomes plain substring search.

**Key intuition:** Either test "same tree?" at every node, or serialize both and ask if one text contains the other.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

---

**Prerequisite knowledge:**
- The Same-Tree comparison.
- Serializing a tree to a string with markers.

## 📝 Problem

Return `True` if `sub` appears as a subtree somewhere inside `root`.

**Example**
```
root = [3,4,5,1,2], sub = [4,1,2] -> True
```

> Two approaches: check-at-every-node `O(m·n)` and serialize-then-substring `O(m+n)`.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Same-Tree at Every Node (worst)

**Idea:** At each node of `root`, test whether the subtree there equals `sub`.

**Time:** `O(m·n)` — an `O(m)` comparison at up to `n` nodes.

**Space:** `O(h)`.

In [ ]:
def is_subtree_brute(root: Optional[TreeNode], sub: Optional[TreeNode]) -> bool:
    if not sub:
        return True                        # an empty tree is a subtree of anything
    if not root:
        return False                       # ran out of tree without a match
    if same_shape(root, sub):              # is the tree hanging here identical to sub?
        return True
    # Otherwise look for sub inside the left or the right subtree.
    return is_subtree_brute(root.left, sub) or is_subtree_brute(root.right, sub)

### Approach 2 — Serialize + Substring (optimal)

**Idea:** Flatten each tree to a string (with markers for values and empties), then check if `sub`'s string sits inside `root`'s string. Markers stop, e.g., 12 from matching a stray 2.

**Time:** `O(m + n)`.

**Space:** `O(m + n)`.

In [ ]:
def is_subtree_serial(root: Optional[TreeNode], sub: Optional[TreeNode]) -> bool:
    def ser(node):                          # flatten a tree to a string, shape included
        if not node:
            return "#"                     # '#' marks an empty child (so shape is captured)
        return "^" + str(node.val) + " " + ser(node.left) + " " + ser(node.right)
    return ser(sub) in ser(root)           # sub is a subtree iff its string sits inside root's

In [ ]:
# Correctness check
r = build_tree([3,4,5,1,2]); s = build_tree([4,1,2])
r2 = build_tree([3,4,5,1,2,None,None,None,None,0]); s2 = build_tree([4,1,2])
tests = [(r,s,True), (r2,s2,False), (build_tree([1,1]), build_tree([1]), True)]
for a, b, exp in tests:
    x, y = is_subtree_brute(a,b), is_subtree_serial(a,b)
    print(f"subtree? brute={x}, serial={y} | expected={exp}")
    assert x == y == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    root = build_balanced(n)
    sub = build_tree([-1, -2, -3])   # not present -> full search
    return (root, sub)
solutions = {
    "brute  O(m*n)": is_subtree_brute,
    "serial O(m+n)": is_subtree_serial,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Reuse a sub-check:** many tree problems are "do this simpler check at every node".
- **Serialize to compare shapes:** flattening a tree to a delimited string turns structure questions into string questions.
- **Signal:** "is one tree contained in another", "matching subtree".
- **Related problems:** Same Tree, Serialize/Deserialize, string matching.
- **Common pitfalls:** (1) no markers → false matches (e.g. 12 vs 2); (2) forgetting empty children in the serialization.